# D2C Robust Evaluation (Google Colab)

This notebook runs a 50-sample evaluation comparing **Vanilla**, **Speech Act (Original)**, and **Speech Act Surgical** using two different judge models (4B and 7B).

### 1. Setup Environment
Install Ollama and project dependencies. We force GPU support by checking nvidia drivers.

In [ ]:
# 1. Install dependencies and Ollama
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import os
import subprocess
import time
import requests

# 2. Set environment variables to help Ollama find CUDA
os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:/usr/local/cuda/lib64' 
os.environ['OLLAMA_DEBUG'] = '1'

print("Verifying NVIDIA drivers...")
!nvidia-smi

print("Starting Ollama server...")
# Start server with nohup to keep it running properly in background
!nohup ollama serve > ollama.log 2>&1 &

time.sleep(20)

# 3. Verify GPU recognition
print("Checking Ollama status...")
try:
    # Try to load a model briefly to force GPU check
    !ollama run qwen3:1.7b "hi" --nowordwrap || echo "First run might pull"
    print("\nOllama Server Logs (Tail):")
    !tail -n 20 ollama.log
except:
    pass

!pip install pandas requests sentence-transformers tqdm

### 2. Pull Models
We use 1.7B for generation and 4B/7B for judging.

In [ ]:
!ollama pull qwen3:1.7b
!ollama pull qwen3.5:4b
!ollama pull qwen2.5:7b

### 3. Project Setup
Ensure you are in the project root.

In [ ]:
print("Current Directory:", os.getcwd())
!ls

### 4. Run Evaluation
This will take approximately 15-20 minutes for 50 samples.

In [ ]:
os.environ['PYTHONPATH'] = os.getcwd()
!python3 -m scripts.colab_detailed_eval

### 5. Download Results

In [ ]:
from google.colab import files
for f in ["colab_detailed_results.csv", "evaluation_audit.txt"]:
    if os.path.exists(f):
        files.download(f)